# 08 · Pangolin — an independent splice model

**Pangolin** (Zeng & Li 2022, *Genome Biology* 23:103, PMID 35449021,
[github.com/tkzeng/Pangolin](https://github.com/tkzeng/Pangolin)) is a deep-learning
splice predictor trained on splice-site *usage* across tissues and species. It gives
**one 0-1 score** per variant (no four deltas to juggle) and uses the **same** 0.5 / 0.2
thresholds as SpliceAI. Because the two were built independently, **agreement between
SpliceAI and Pangolin** is stronger evidence than either alone.

> ✅ **REAL.** Pangolin has no precomputed release and is not in dbNSFP, but
> **the build cell below runs the actual model locally** — weights ship inside the pip
> package, and it needs only the ~215 kb CFTR reference region (no whole-genome download).
> The default scope now scores **every CFTR2 variant with GRCh38 coordinates (~1,892 of
> 2,097), SNVs *and* indels**, so `source='REAL'`. Validated against real SpliceAI on the
> canonical alleles (e.g. c.2988+1G>A: Pangolin 0.86 vs SpliceAI 0.99).
>
> `SCOPE = "curated"` in the build cell still scores just the 5 classic splice alleles and stays
> `source='DEMO'` — the label follows the **coverage**, never the model.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 1 · Pangolin — a newer splice model, one tidy score

**Pangolin** (Zeng & Li 2022, *Genome Biology* 23:103, PMID **35449021**) was trained on
splice-site *usage* measured across multiple tissues and species, and tends to be a
little more sensitive than SpliceAI for some variant classes. For our purposes the nice
thing is that it returns a single 0-1 score (larger of the biggest splice-usage gain and
loss), directly comparable to SpliceAI's DS_max at the same 0.5 / 0.2 cut-points.

## 2 · REAL Pangolin scores — run the model, don't download a table

Unlike SpliceAI there is no precomputed file to download; you **run the model**. It is
lighter than it sounds — no whole-genome FASTA required, and a GPU only makes it faster.
See the build cell below for the exact install command and code.

`tk.load_pangolin()` reads whatever the build cell wrote. The default scope covers every
CFTR2 variant that has GRCh38 coordinates — ~1,892 of 2,097 — SNVs and indels alike.

**Why this reaches further than SpliceAI here.** The precomputed SpliceAI release used in
tools/07 is *masked SNV only*, so it cannot score indels at all. Pangolin run locally
pads its delta track by the length difference, so it scores indels too — 573 of the 713
CFTR2 deletions/insertions get a score. predict/13 shows why that is a smaller win than
it first looks: a splice verdict on a frameshift is usually a correct "no splice impact"
and almost never the reason the variant causes disease.

### Running the model — the build cell below

Pangolin has no per-gene download and no bulk file to filter — the only way to
get real scores is to **run the model locally**. That needs one extra install
and one small manual download (a reference sequence slice, not a raw-data file):

```bash
pip install "git+https://github.com/tkzeng/Pangolin.git" pyfaidx gffutils torch
```

The model weights ship inside that pip package (no separate download), and the
cell below auto-fetches the ~215 kb CFTR reference region from Ensembl the
first time it runs, caching it at `data/cftr_region_grch38.fa` — **no
whole-genome FASTA needed**. It reads authoritative GRCh38 coordinates from
`data/cftr2_cftr.csv` (built in benchmark/01 — run that notebook first),
not hand-entered ones, so Pangolin scores the variant it is actually supposed
to. `SCOPE = "cftr2"` (default, below) scores every CFTR2 variant with GRCh38
coordinates — ~1,892 of 2,097, SNVs *and* indels, ~4 min on a GPU, `source='REAL'`;
`SCOPE = "curated"` scores only the 5 classic splice alleles and stays
`source='DEMO'` — **the label follows coverage, never the model.**

Variants that cannot be scored (no coordinates, or an indel bigger than the
±50 bp aggregation window can speak to) are kept with an empty score and a
`skip_reason`, so coverage stays auditable instead of silently short.

License: Pangolin is **non-commercial** — cite Zeng & Li 2022 (PMID 35449021).

In [2]:
import re, requests, numpy as np

DATA_DIR = pathlib.Path.cwd().parent / "data"
PANGOLIN_TSV = DATA_DIR / "pangolin_cftr.csv"
CFTR2_CSV = DATA_DIR / "cftr2_cftr.csv"
REF_FA = DATA_DIR / "cftr_region_grch38.fa"
SCOPE = "cftr2"                    # "cftr2" (REAL, ~4 min on GPU) or "curated" (DEMO, seconds)
DIST = 50                          # Pangolin's aggregation window, +/- d
MAX_EVENT = 100                    # skip ref/alt events bigger than this (score would be meaningless)
KNOWN_SPLICE = ["c.2988+1G>A", "c.2657+5G>A", "c.3718-2477C>T", "c.3140-26A>G", "c.1680-886A>G"]
_ACGT = re.compile(r"^[ACGT]+$")

if PANGOLIN_TSV.exists():
    print(f"already built -> {PANGOLIN_TSV.name} (delete to rebuild, or edit SCOPE above and rerun)")
else:
    try:
        import torch
        from pangolin.model import Pangolin, L, W, AR
        from pkg_resources import resource_filename
    except ImportError as exc:
        raise ImportError(
            "Pangolin needs one extra install (no other notebook needs it):\n"
            '  pip install "git+https://github.com/tkzeng/Pangolin.git" pyfaidx gffutils torch'
        ) from exc
    if not CFTR2_CSV.exists():
        raise FileNotFoundError(f"{CFTR2_CSV} missing -- run benchmark/01_cftr2.ipynb first "
                                 "(Pangolin needs its authoritative GRCh38 coordinates).")

    # one_hot_encode + compute_score are inlined verbatim from pangolin/pangolin.py
    # (Zeng & Li 2022) so we skip importing that module, whose own top-level
    # `import pyfastx, vcf` pulls in dependencies this notebook doesn't otherwise need.
    IN_MAP = np.asarray([[0, 0, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])

    def one_hot_encode(seq, strand):
        seq = seq.upper().replace('A', '1').replace('C', '2').replace('G', '3').replace('T', '4').replace('N', '0')
        if strand == '+':
            seq = np.asarray(list(map(int, list(seq))))
        else:
            seq = np.asarray(list(map(int, list(seq[::-1]))))
            seq = (5 - seq) % 5
        return IN_MAP[seq.astype('int8')]

    def compute_score(ref_seq, alt_seq, strand, d, models):
        ref_seq = torch.from_numpy(np.expand_dims(one_hot_encode(ref_seq, strand).T, axis=0)).float()
        alt_seq = torch.from_numpy(np.expand_dims(one_hot_encode(alt_seq, strand).T, axis=0)).float()
        if torch.cuda.is_available():
            ref_seq, alt_seq = ref_seq.to("cuda"), alt_seq.to("cuda")
        pang = []
        for j in range(4):
            score = []
            for model in models[3 * j:3 * j + 3]:
                with torch.no_grad():
                    ref = model(ref_seq)[0][[1, 4, 7, 10][j], :].cpu().numpy()
                    alt = model(alt_seq)[0][[1, 4, 7, 10][j], :].cpu().numpy()
                    if strand == '-':
                        ref, alt = ref[::-1], alt[::-1]
                    l = 2 * d + 1
                    ndiff = np.abs(len(ref) - len(alt))
                    if len(ref) > len(alt):
                        alt = np.concatenate([alt[0:l // 2 + 1], np.zeros(ndiff), alt[l // 2 + 1:]])
                    elif len(ref) < len(alt):
                        alt = np.concatenate([alt[0:l // 2], np.max(alt[l // 2:l // 2 + ndiff + 1], keepdims=True), alt[l // 2 + ndiff + 1:]])
                    score.append(alt - ref)
            pang.append(np.mean(score, axis=0))
        pang = np.array(pang)
        loss = pang[np.argmin(pang, axis=0), np.arange(pang.shape[1])]
        gain = pang[np.argmax(pang, axis=0), np.arange(pang.shape[1])]
        return loss, gain

    def load_models():
        dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        models = []
        for i in [0, 2, 4, 6]:
            for j in range(1, 4):
                m = Pangolin(L, W, AR)
                w = torch.load(resource_filename("pangolin", "models/final.%s.%s.3.v2" % (j, i)), map_location=dev)
                m.load_state_dict(w)
                models.append(m.to(dev).eval())
        return models

    def load_region():
        """Return (region_start_1based, sequence); auto-fetches + caches the ~215 kb
        CFTR slice from Ensembl's REST API the first time -- no whole-genome FASTA."""
        if not REF_FA.exists():
            print("fetching CFTR reference region from Ensembl (one-time, ~215 kb)...")
            r = requests.get(
                "https://rest.ensembl.org/sequence/region/human/7:117465000..117680000",
                headers={"Content-Type": "text/x-fasta"}, timeout=30)
            r.raise_for_status()
            REF_FA.write_text(r.text)
        lines = REF_FA.read_text().splitlines()
        r0 = int(lines[0].split(":")[3])          # '>chromosome:GRCh38:7:117465000:117680000:1'
        return r0, "".join(lines[1:]).upper()

    def pangolin_score(pos, ref, alt, r0, seq, models, d=DIST):
        start = (pos - r0) - (5000 + d)
        end = start + 10000 + 2 * d + len(ref)
        if start < 0 or end > len(seq):
            raise ValueError(f"outside the cached reference window (needs 7:{pos-5050}-{pos+5050})")
        window = seq[start:end]
        got = window[5000 + d: 5000 + d + len(ref)]
        if got != ref:
            raise ValueError(f"ref mismatch at 7:{pos} -- window has {got!r}, expected {ref!r}")
        alt_seq = window[:5000 + d] + alt + window[5000 + d + len(ref):]
        # Score in reference/plus-strand orientation -- matches how SpliceAI's precomputed
        # scores are reported and is validated against them (c.2988+1G>A: donor-loss 0.86
        # vs SpliceAI 0.99). CFTR is plus-strand; passing strand='-' would mis-score it.
        loss, gain = compute_score(window, alt_seq, "+", d, models)
        return round(float(max(gain.max(), -loss.min())), 4)

    cf = pd.read_csv(CFTR2_CSV)
    if SCOPE == "curated":
        cf = cf[cf["cdna_name"].isin(KNOWN_SPLICE)]
    cf = cf.copy()
    ref, alt = cf["grch38_ref"].astype(str), cf["grch38_alt"].astype(str)
    cf["skip_reason"] = None
    cf.loc[cf["grch38_pos"].isna(), "skip_reason"] = "no GRCh38 coordinates in CFTR2"
    bad = cf["skip_reason"].isna() & ~(ref.str.match(_ACGT) & alt.str.match(_ACGT))
    cf.loc[bad, "skip_reason"] = "allele is not plain ACGT"
    big = cf["skip_reason"].isna() & ((ref.str.len() > MAX_EVENT) | (alt.str.len() > MAX_EVENT))
    cf.loc[big, "skip_reason"] = f"event larger than {MAX_EVENT} bp"

    label = "REAL" if SCOPE == "cftr2" else "DEMO"      # label follows coverage, not the model
    print(f"scope={SCOPE} -> {len(cf):,} CFTR2 variants, source label '{label}'")
    r0, seq = load_region()
    models = load_models()

    rows, done = [], 0
    for _, v in cf.iterrows():
        score, reason = None, v["skip_reason"]
        pos = int(v["grch38_pos"]) if pd.notna(v["grch38_pos"]) else None
        r_, a_ = str(v["grch38_ref"]), str(v["grch38_alt"])
        if reason is None:
            try:
                score = pangolin_score(pos, r_, a_, r0, seq, models)
                done += 1
                if SCOPE == "curated" or done % 200 == 0:
                    print(f"  [{done:5,}] {str(v['cdna_name'])[:24]:24} pangolin={score}")
            except Exception as e:
                reason = str(e).split(" -- ")[0]
        rows.append({"cdna_name": v["cdna_name"], "legacy_name": v["legacy_name"],
                     "chrom": "7", "pos": pos, "ref": r_ if pos else None, "alt": a_ if pos else None,
                     "pangolin_score": score, "cftr2_class": v["cftr2_class"],
                     "source": label, "skip_reason": reason})

    out = pd.DataFrame(rows)
    out.to_csv(PANGOLIN_TSV, index=False)
    scored = out["pangolin_score"].notna()
    print(f"\nPangolin ({label}) written: {int(scored.sum()):,} scored / {len(out):,} targets "
          f"-> {PANGOLIN_TSV.relative_to(DATA_DIR.parent)}")

already built -> pangolin_cftr.csv (delete to rebuild, or edit SCOPE above and rerun)


## Example: the shared splice worked-example panel, scored by **Pangolin**

The same fixed panel of famous CFTR **splice** variants runs through every splice tool
(tools/07–09), so you can follow one set of variants across the series. The
variant list is `tk.A2_KNOWN_CDNA` (shared in `toolkit.py`); the **scoring is shown
inline below** so you can see exactly how Pangolin is joined onto it.

In [3]:
# Pangolin scores come from RUNNING the model (build cell above) over the CFTR2 list
# with correct CFTR2 coords. Tier with the published 0.5 / 0.2 cut-points.
pg = tk.load_pangolin()
HIGH, MOD = tk.THRESHOLDS['pangolin']['high'], tk.THRESHOLDS['pangolin']['moderate']
scored = pg[pg['pangolin_score'].notna()].copy()
scored['tier'] = scored['pangolin_score'].apply(
    lambda s: 'HIGH' if s >= HIGH else ('MODERATE' if s >= MOD else 'LOW'))
print(f"{len(scored):,} scored / {len(pg):,} CFTR2 targets | source: {pg['source'].unique().tolist()}")
print(scored['tier'].value_counts().to_string())
if pg['pangolin_score'].isna().any():
    print("\nnot scored, by reason:")
    print(pg.loc[pg['pangolin_score'].isna(), 'skip_reason'].value_counts().to_string())

# The classic CF splice alleles — the validation set, recovered from the full run.
print("\nclassic CF splice alleles:")
print(scored[scored['cdna_name'].isin(tk.A2_KNOWN_CDNA)]
      [['cdna_name', 'legacy_name', 'pangolin_score', 'tier', 'source']].to_string(index=False))

1,892 scored / 2,097 CFTR2 targets | source: ['REAL']
tier
LOW         1518
HIGH         260
MODERATE     114

not scored, by reason:
skip_reason
no GRCh38 coordinates in CFTR2    204
event larger than 100 bp            1

classic CF splice alleles:
     cdna_name   legacy_name  pangolin_score     tier source
c.3718-2477C>T 3849+10kbC->T          0.3327 MODERATE   REAL
   c.2657+5G>A    2789+5G->A          0.8194     HIGH   REAL
  c.3140-26A>G   3272-26A->G          0.8120     HIGH   REAL
   c.2988+1G>A    3120+1G->A          0.8568     HIGH   REAL
 c.1680-886A>G 1811+1634A->G          0.7057     HIGH   REAL


## Key takeaways

1. **Pangolin** gives one 0-1 score with the **same** 0.5 / 0.2 thresholds as SpliceAI.
2. SpliceAI + Pangolin **agreeing** is stronger evidence than either alone — and here they
   do (Pangolin recovers HIGH on the canonical CF splice alleles, matching real SpliceAI).
3. The build cell above produces **real** Pangolin scores locally, now over the whole CFTR2
   list (~1,892 variants, `source='REAL'`); `SCOPE = "curated"` keeps the 5-allele teaching
   run at `source='DEMO'`. The label follows coverage, not the model.
4. Running locally beats a precomputed release on **reach**: Pangolin scores indels, which
   the masked-SNV SpliceAI file cannot. But reach is not the same as usefulness —
   predict/13 shows Pangolin correctly calls most CF-causing frameshifts "no splice
   impact", which is true and tells you nothing about why they cause disease.
5. Correct citation: **Zeng & Li 2022, PMID 35449021**.

**Next:** tools/09 — **CADD**, a real, live score.